# Simple MCP Server.
We'll create a simple MCP server that allows an AI agent to search for files on your local Windows computer. First, create a new Python Project and implement a virtual environment using uv. Inside this new project, create another folder called "test-files" and add the following dummy files: notes.txt, cybersecurity.txt, and python_basics.py

In [ ]:
touch notes.txt cybersecurity.txt pyton_basics.py

In your project root, create a new Python file named server.py. Paste the following code into it, and run it to add the required modules (uv add "mcp[cli]"):

In [ ]:
from pathlib import Path
from mcp.server import MCPServer

# Create our MCP server
mcp = MCPServer("Local File Search")

# Define the folder we are allowed to search
BASE_DIR = Path(__file__).resolve().parent / "test_files"

@mcp.tool()
def list_files() -> list[str]:
    """List all files in the test_files directory."""

    files = []

    for file in BASE_DIR.iterdir():
        if file.is_file():
            files.append(file.name)

    return files

if __name__ == "__main__":
    mcp.run()


The @mcp.tool() decorator tells FastMCP:
"Make the following Python function available as a tool that an MCP client can discover and call."
Without this decorator, list_files() would simply be a regular Python function. The -> list[str] is a type hint indicating that the function returns a list of strings.
Because MCP uses standard input/output communication by default, the server may appear to do nothing. That is normal. It is waiting for an MCP client to communicate with it. You can stop it with Ctrl+C. In our next step, we'll use the MCP Inspector to actually call the list_files() tool and see its results.
We will use MCP Inspector, an official debugging tool that allows us to communicate with an MCP server without connecting an AI model yet.


In [ ]:
uv run mcp dev server.py

 After running the command, your terminal should display a local URL. Open that URL in your browser.
 Once you are connected:
1. Select the Tools tab.
2. Click List Tools, if necessary.
3. Find the tool named list_files.
4. Select it and click Run Tool.
If everything works, you should receive something similar to:
[
  "notes.txt",
  "cybersecurity.txt",
  "python_basics.py"
]   

## Create a tool to search for filenames
Now let's give our MCP server a second capability.
Instead of listing every file, we want to search for files using part of their names.
For example, searching for python should return python_basics.py.
Add our new tool:

In [ ]:
@mcp.tool()
def search_files(keyword: str) -> list[str]:
    """Search for filenames containing a keyword."""

    matches = []

    for file in BASE_DIR.iterdir():
        if file.is_file() and keyword.lower() in file.name.lower():
            matches.append(file.name)

    return matches


Test our new tool
After saving server.py, reconnect in the Inspector if necessary.
1. Select the new search_files tool.
2. Enter python in the keyword field and click Run Tool.
You should receive:
{
  "result": [
    "python_basics.py"
  ]
}

## Search inside files
Now we'll make our MCP server more useful.
Currently, we have two tools:
- list_files() — Lists all filenames.
- search_files() — Searches for a keyword in filenames.
Our third tool will be search_content(), which searches for a word inside files.

Add content to the file cybersecurity.txt:
Cybersecurity protects computers and networks.
Python is useful for security automation.
MCP allows AI agents to access external tools.

Add our third tool:


In [ ]:

@mcp.tool()
def search_content(keyword: str) -> list[str]:
    """Search for a keyword inside text files."""

    matches = []

    for file in BASE_DIR.iterdir():
        if file.is_file() and file.suffix in [".txt", ".py", ".md"]:
            
            # This opens the file, reads its contents as text, and stores them in the variable content:
            content = file.read_text(encoding="utf-8")

            if keyword.lower() in content.lower():
                matches.append(file.name)

    return matches


Test our new tool.
Restart MCP Inspector.
Find search_content, enter MCP as the keyword, and run the tool.
Your expected structured output is:
{
  "result": [
    "cybersecurity.txt"
  ]
}


This assumes no other test file contains MCP.

## Display matching lines and line numbers
Currently, search_content() tells us which file contains a keyword, but not where the keyword appears.
Let's improve this. For example, instead of returning only:
cybersecurity.txt

We want something like:
cybersecurity.txt (Line 3): MCP allows AI agents to access external tools.

In server.py, replace your current search_content() function with this version:

In [ ]:

@mcp.tool()
def search_content(keyword: str) -> list[str]:
    """Search inside files and return matching lines."""

    matches = []

    for file in BASE_DIR.iterdir():
        if file.is_file() and file.suffix in [".txt", ".py", ".md"]:

            content = file.read_text(encoding="utf-8")
            
            ''' splitlines() divides the file's contents into individual lines.
                enumerate() lets us access each line along with its line number.
                For example:
                for number, line in enumerate(["Hello", "Python", "MCP"], start=1):
                    print(number, line)
                Output:
                1 Hello
                2 Python
                3 MCP

                Notice that start=1 makes the numbering begin at 1 rather than 0.
            '''
                for line_number, line in enumerate(content.splitlines(), start=1):
                if keyword.lower() in line.lower():
                    matches.append(
                        f"{file.name} (Line {line_number}): {line.strip()}"
                    )

    return matches


## Test the modified tool
Restart MCP Inspector, select search_content, and enter MCP as the keyword.

Assuming your cybersecurity.txt contains our earlier example, the result should resemble:

{
  "result": [
    "cybersecurity.txt (Line 3): MCP allows AI agents to access external tools."
  ]
}

## Add error handling
Before connecting our MCP server to an AI agent, let's make it a little more reliable.
if a file contains characters that cannot be decoded using UTF-8
Python may raise a UnicodeDecodeError, causing our tool call to fail.
In server.py, replace your current search_content() function with this version:

In [ ]:

@mcp.tool()
def search_content(keyword: str) -> list[str]:
    """Search inside files and return matching lines."""

    matches = []

    for file in BASE_DIR.iterdir():
        if file.is_file() and file.suffix in [".txt", ".py", ".md"]:

            try:
                content = file.read_text(encoding="utf-8")

                for line_number, line in enumerate(content.splitlines(), start=1):
                    if keyword.lower() in line.lower():
                        matches.append(
                            f"{file.name} (Line {line_number}): {line.strip()}"
                        )

            ''' except tells Python what to do if an expected error occurs.
                In this case, we're handling two types of errors:
                Error	                    Meaning
                UnicodeDecodeError	Python cannot interpret the file's text encoding.
                OSError	            Python encounters a filesystem problem, such as a permission error.
                Instead of allowing one problematic file to stop the entire search, our tool records the problem
                and continues searching other files.
            '''
            except (UnicodeDecodeError, OSError):
                matches.append(f"Could not read: {file.name}")

    return matches


## Here is our complete server:

In [ ]:
from pathlib import Path
from mcp.server import MCPServer

# Create our MCP server
mcp = MCPServer("Local File Search")

# Define the folder we are allowed to search
BASE_DIR = Path(__file__).resolve().parent / "test_files"

@mcp.tool()
def list_files() -> list[str]:
    """List all files in the test_files directory."""

    files = []

    for file in BASE_DIR.iterdir():
        if file.is_file():
            files.append(file.name)

    return files

@mcp.tool()
def search_files(keyword: str) -> list[str]:
    """Search for filenames containing a keyword."""

    matches = []

    for file in BASE_DIR.iterdir():
        if file.is_file() and keyword.lower() in file.name.lower():
            matches.append(file.name)

    return matches

@mcp.tool()
def search_content(keyword: str) -> list[str]:
    """Search inside files and return matching lines."""

    matches = []

    for file in BASE_DIR.iterdir():
        if file.is_file() and file.suffix in [".txt", ".py", ".md"]:

            try:
                content = file.read_text(encoding="utf-8")

                for line_number, line in enumerate(content.splitlines(), start=1):
                    if keyword.lower() in line.lower():
                        matches.append(f"{file.name} (Line {line_number}): {line.strip()}")

            except (UnicodeDecodeError, OSError):
                matches.append(f"Could not read: {file.name}")

    return matches

if __name__ == "__main__":
    mcp.run()


# Prepare to connect our MCP server to an AI agent
This is where our project becomes more interesting!
Until now, we've used MCP Inspector to select and execute tools manually. Our next goal is to let an LLM decide which tool to execute, just as we did in the VirusTotal MCP project.

# Create the MCP client file
In your existing project, create a new empty Python file named client.py

## Connect our Python client to the MCP server
For this step, we're going to do only one thing: establish a connection between client.py and server.py.
We won't call LiteLLM yet. First, we need to verify that our client can discover the three tools we created.
Once that works, we'll add the LLM in the next step.

stdio means standard input and standard output. Our client will communicate with the server through a local process, not through an internet request.

Open the empty client.py file you created earlier and paste in the following:

Note: The following script follows the connection pattern demonstrated in the official MCP Python SDK client examples.

In [ ]:

import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Configure how to start our MCP server.
# It's equivalent to running the following command in your terminal:
# uv run server.py
# Note: command expects a string, while args expects a list of strings.
server_params = StdioServerParameters(command="uv", args=["run", "server.py"])

# The keyword async tells Python: "This is an asynchronous function, also called a coroutine function."
# Asynchronous programming allows Python to manage that waiting efficiently.
async def main():

    # Start the server and establish stdio communication
    # This launches the server as a separate process and opens two communication streams:
    # - read — Receives messages from the MCP server.
    # - write — Sends messages to the MCP server.
    # Notice that read and write are named from the client's perspective.
    # stdio_client() is a function provided by the MCP Python SDK.
    async with stdio_client(server_params) as (read, write):

        # Create an MCP client session
        async with ClientSession(read, write) as session:

            # Initialize the MCP connection
            await session.initialize()

            print("Connected to MCP server!")

            # Ask the server which tools are available.
            # Our client doesn't need to know in advance what tools the server provides.
            # It asks the server for that information.
            tools = await session.list_tools()

            # Why do we write tools.tools instead of simply tools?
            # Because the first tools is our Python variable, and the second .tools is an attribute of the response object.
            for tool in tools.tools:
                print(f"Available tool: {tool.name}")


if __name__ == "__main__":
    asyncio.run(main())


asyncio is a built-in Python library that provides the machinery for running asynchronous code.

Let's compare:

Term	Purpose

async	Defines an asynchronous function or is used in async with and async for.

asyncio	Python library for managing asynchronous operations.

asyncio.run()	Starts an event loop, executes a coroutine, and cleans up afterward.

await	Suspends the current coroutine while waiting for an awaitable operation to finish.


- stdio_client() establishes the communication channels.
  
- ClientSession() uses those channels to exchange MCP messages.

  
- session.initialize() performs the initial MCP handshake with the server.

After initialization, we can use the session to request available tools or execute them.

## Understanding how LiteLLM receives MCP tools.

When we run:

tools = await session.list_tools()

The MCP server returns objects containing information about its tools. For example, a simplified MCP tool definition looks like:

{

    "name": "search_files",
     "description": "Search for filenames containing a keyword.",
    "inputSchema": {
        "type": "object",
        "properties": {
            "keyword": {"type": "string"}
        },
        "required": ["keyword"]
    }
}

However, LiteLLM's OpenAI-compatible tool-calling interface expects this structure:

{
    
    "type": "function",
    "function": {
        "name": "search_files",
        "description": "Search for filenames containing a keyword.",
        "parameters": {
            "type": "object",
            "properties": {
                "keyword": {"type": "string"}
            },
            "required": ["keyword"]
        }
    }
}


## Let's write the conversion code
For now, we'll make a small modification to client.py. After:

tools = await session.list_tools()

Add:

In [ ]:

# Convert MCP tool definitions to LLM-compatible format
llm_tools = []

for tool in tools.tools:
    llm_tools.append({
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description or "",
            "parameters": tool.input_schema
        }
    })

# Display the converted tool definitions
for tool in llm_tools:
    print(tool)


We can use:

import json
    
print(json.dumps(tool, indent=2))

For better visualization.

## Make our first LiteLLM request
Now we're reaching one of the most interesting parts of our MCP project: allowing an LLM to decide which tool to use.

One important detail: the LLM doesn't directly access your local files. Our Python MCP client controls which tool calls are executed.

Create the file .env and add the key

Add our first LiteLLM request. At the top of client.py, add:

In [ ]:
import os
from litellm import acompletion

Also add this code:

In [ ]:
# Prepare our question for the LLM
messages = [
    {
        "role": "user",
        "content": "Which files contain the word cybersecurity?"
    }
]

# Send the question and available tools to the LLM
response = await acompletion(
    model="openai/gpt-4.1-mini",
    api_key=os.environ["OPENAI_API_KEY"],
    messages=messages,
    tools=llm_tools,
    tool_choice="auto"
)

# Inspect the LLM's response
message = response.choices[0].message

print("\nLLM Response:")
print(message.model_dump_json(indent=2))


In [ ]:
The output will be somthing like:

In [ ]:
LLM Response:
{
  "content": null,
  "role": "assistant",
  "tool_calls": [
    {
      "function": {
        "arguments": "{\"keyword\":\"cybersecurity\"}",
        "name": "search_files"
      },
      "id": "call_I289E3fHo99Ff2MtrmcC5Yhk",
      "type": "function"
    }
  ],
  "function_call": null,
  "provider_specific_fields": {
    "refusal": null
  },
  "annotations": []
}

This tells us two things:

- Tool selected: search_files
- Argument: keyword="cybersecurity"
  
The LLM correctly generated a tool-call request with the required parameter.

## Execute the tool selected by the LLM
Now let's make our agent actually execute the selected MCP tool.

Previously, the LLM only told us which function it wanted to call.

This time, our client will receive that request and send it to the MCP server.

Add this code to client.py

In [ ]:

# Check whether the LLM requested a tool
if message.tool_calls:

    for tool_call in message.tool_calls:

        # Extract the tool name
        tool_name = tool_call.function.name

        # Convert the JSON arguments into a Python dictionary
        tool_args = json.loads(tool_call.function.arguments)

        print(f"\nCalling MCP tool: {tool_name}")
        print(f"Arguments: {tool_args}")

        # Execute the tool through our MCP session
        result = await session.call_tool(
            name=tool_name,
            arguments=tool_args
        )

        # Display the result returned by the MCP server
        print("\nMCP Tool Result:")
        print(result.model_dump_json(indent=2))


## Send the MCP result back to the LLM
We're almost finished building our first AI agent!

Currently, our program stops after receiving the result from the MCP server.

We want the LLM to use that result to generate a natural-language answer.

In [ ]:

    # Add the LLM's tool-call request to the conversation
    # The method: message.model_dump()
    # Converts that object into a regular Python dictionary.
    messages.append(message.model_dump(exclude_none=True))

    # Add each MCP tool result to the conversation
    # (For this first test, we use one tool call.)
    messages.append({
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": result.model_dump_json()
    })

    # Ask the LLM to generate its final answer
    final_response = await acompletion(
        model="openai/gpt-4.1-mini",
        api_key=os.environ["OPENAI_API_KEY"],
        messages=messages
    )

    print("\nFinal AI Answer:")
    print(final_response.choices[0].message.content)


One importat note:

message.model_dump()

Converts an object into a Python dictionary.

result.model_dump_json()

Converts an object into a JSON-formatted string.

## Allow the LLM to call multiple MCP tools
Currently, our client is designed around a single tool call. But an LLM might need more than one tool to answer a question.

Replace that entire section, including the final acompletion() request, with the following:

In [ ]:

# Check whether the LLM requested any tools
if message.tool_calls:

    # Save the assistant's tool requests
    messages.append(message.model_dump(exclude_none=True))

    # Execute each requested tool
    for tool_call in message.tool_calls:

        tool_name = tool_call.function.name
        tool_args = json.loads(tool_call.function.arguments)

        print(f"\nCalling MCP tool: {tool_name}")
        print(f"Arguments: {tool_args}")

        # Execute the requested MCP tool
        result = await session.call_tool(
            name=tool_name,
            arguments=tool_args
        )

        # Display the result
        print("\nMCP Tool Result:")
        print(result.model_dump_json(indent=2))

        # Add this tool's result to the conversation
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": result.model_dump_json()
        })

    # Ask the LLM to interpret all tool results
    final_response = await acompletion(
        model="openai/gpt-4.1-mini",
        api_key=os.environ["OPENAI_API_KEY"],
        messages=messages
    )

    print("\nFinal AI Answer:")
    print(final_response.choices[0].message.content)

else:
    print("\nFinal AI Answer:")
    print(message.content)


With the previous code we noticed that there's a problem with the final answer. Only one tool is executed. That means the LLM's list of all three filenames is not supported by the tool output shown. Even though the filenames happen to match our test folder, the model shouldn't claim to have listed the directory without using the appropriate tool.
This is an example of why we shouldn't always trust an LLM's response without checking the underlying tool results.

Improve our tool descriptions.

Before building a more complex agent loop, I suggest we make our MCP tools easier for the LLM to distinguish.
In server.py, update the three function docstrings as follows.

Tool 1: list_files()

In [ ]:
"""List all filenames in the test_files directory.
Use this tool when the user asks which files exist
or requests a complete directory listing.
"""

In [ ]:
Tool 2: search_files()

In [ ]:
"""Search filenames for a keyword.
Use this tool only when searching file NAMES,
not when searching the contents of files.
"""

In [ ]:
Tool 3: search_content()

In [ ]:
"""Search inside text files for a keyword.
Returns matching filenames, line numbers, and text.
Use when the user asks which files contain a word
or phrase in their contents.
"""

## Creating our agent loop
Instead of allowing unlimited LLM requests, we'll give our agent a maximum of 5 rounds.

Now let's modify client.py to allow the LLM to request additional tools when necessary.

We'll replace our previous two-request approach with a loop.

Find this section in client.py 


In [ ]:
# Send the question and available tools to the LLM
response = await acompletion(

We will replace everything from that section through the final-answer handling code. 

Keep your imports, MCP connection, tool conversion, and messages definition unchanged.

In [ ]:

# Maximum number of LLM requests
max_rounds = 5

for round_number in range(max_rounds):

    print(f"\nAgent Round: {round_number + 1}")

    # Ask the LLM what to do next
    response = await acompletion(
        model="openai/gpt-4.1-mini",
        api_key=os.environ["OPENAI_API_KEY"],
        messages=messages,
        tools=llm_tools,
        tool_choice="auto"
    )

    message = response.choices[0].message

    # If the LLM doesn't request tools, finish
    if not message.tool_calls:
        print("\nFinal AI Answer:")
        print(message.content)
        break

    # Save the assistant's tool requests
    messages.append(message.model_dump(exclude_none=True))

    # Execute all requested MCP tools
    for tool_call in message.tool_calls:

        tool_name = tool_call.function.name
        tool_args = json.loads(tool_call.function.arguments)

        print(f"\nCalling MCP tool: {tool_name}")
        print(f"Arguments: {tool_args}")

        result = await session.call_tool(
            name=tool_name,
            arguments=tool_args
        )

        print("\nMCP Tool Result:")
        print(result.model_dump_json(indent=2))

        # Save each result in conversation history
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": result.model_dump_json()
        })

else:
    print("\nMaximum number of agent rounds reached.")
    print("The agent stopped without generating a final answer.")


Here is the file client.py complete:

In [ ]:
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
import json
import os
from litellm import acompletion

# Configure how to start our MCP server
server_params = StdioServerParameters(
    command="uv", args=["run", "server.py"])

async def main():

    # Start the *** SERVER *** and establish stdio communication
    async with stdio_client(server_params) as (read, write):

        # Create an MCP ** CLIENT ** session
        async with ClientSession(read, write) as session:

            # Initialize the MCP connection ** CLIENT - SERVER **
            await session.initialize()

            print("Connected to MCP server!")
            print('*'*100)

            # Ask the server which tools are available
            tools = await session.list_tools()
            print(f"This is the server response with the avail tools\n {tools.model_dump_json(indent=2)}")
            input("Enter to continue")
            # Convert MCP tool definitions to LLM-compatible format
            llm_tools = []

            for tool in tools.tools:
                llm_tools.append({"type": "function", "function": {
                    "name": tool.name,
                    "description": tool.description or "",
                    "parameters": tool.input_schema}})

            # Display the extracted tool definitions
            for tool in llm_tools:
                print(f"Here is the information about tools extracted from tools\n {json.dumps(tool, indent=2)}")
                print('*'*100)
                input('Enter to continue')
            # Prepare our question for the LLM
            messages = [{
                "role": "user",
                "content": "List all the files and find which ones contain the word Python."
                }]

            # Maximum number of LLM requests
            max_rounds = 5
            for round_number in range(max_rounds):
                print(f"\nAgent Round: {round_number + 1}")

                # Send the question and available tools to the LLM
                response = await acompletion(
                    model="openai/gpt-4.1-mini",
                    api_key=os.environ["OPENAI_API_KEY"],
                    messages=messages,
                    tools=llm_tools,
                    tool_choice="auto")
            
                print(f"Here is the full content of the LLM response:\n{response.model_dump_json(indent=2)}")
                print('*'*100)
                input('Enter to continue')

                # Inspect the LLM's response
                message = response.choices[0].message

                 # If the LLM doesn't request tools, finish
                if not message.tool_calls:
                    print("\nFinal AI Answer:")
                    print(message.content)
                    break
                # Save the assistant's tool requests
                messages.append(message.model_dump(exclude_none=True))

                # Execute all requested MCP tools
                for tool_call in message.tool_calls:
                    # Extract the tool name
                    tool_name = tool_call.function.name

                    # Convert the JSON arguments into a Python dictionary
                    tool_args = json.loads(tool_call.function.arguments)

                    print(f"\nCalling MCP tool: {tool_name}")
                    print(f"Arguments: {tool_args}")

                    # Execute the tool through our MCP session
                    result = await session.call_tool(
                        name=tool_name,
                        arguments=tool_args)

                    # Display the result returned by the MCP server
                    print('*'*100)
                    print("\nMCP Tool Result (Executes the tool (tool name and the arguments)):")
                    print(result.model_dump_json(indent=2))
                    
                    # Add each MCP tool result to the conversation
                    # (For this first test, we use one tool call.)
                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "content": result.model_dump_json()})
                else:
                    print("\nMaximum number of agent rounds reached.")
                    print("The agent stopped without generating a final answer.")

                # Ask the LLM to generate its final answer
                final_response = await acompletion(
                    model="openai/gpt-4.1-mini",
                    api_key=os.environ["OPENAI_API_KEY"],
                    messages=messages)

                print("\nFinal AI Answer:")
                print(final_response.choices[0].message.content)

            else:
                print("\nFinal AI Answer:")
                print(message.content)

if __name__ == "__main__":
    asyncio.run(main())
